In [1]:
!pip install -q \
  transformers \
  huggingface_hub \
  evaluate \
  spacy \
  accelerate

!pip -q install "gradio==6.0.2"
!pip install -U bitsandbytes
!python -m spacy download de_core_news_md

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 MB 17.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import re
import torch
from peft import PeftModel
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import hf_hub_download, login
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from typing import List, Dict
import gradio as gr
import spacy
import time

In [3]:
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
@dataclass
class SegmentInfo:
    text: str         # clause/sentence text
    label: str        # "negative" | "neutral" | "positive"
    score: float      # numeric score in [-1, 1]
    length: int       # len(text)

# Class for saving data of one conversation turn
@dataclass
class TurnInfo:
    who: str           # "human" | "bot"
    text: str          # Current sent text
    s_raw: float       # per-turn sentiment in [-1..1] (overall_score)
    s_ema: float       # smoothed per-turn sentiment
    trend: str         # Current trend: "up"|"down"|"flat"
    event: str | None  # "reversal_to_up"/"reversal_to_down"/None
    desired: str       # Desired trend: "up"|"down"|"flat"
    overall_label: str # "negative" | "neutral" | "positive"
    segments: List[SegmentInfo]  # fine-grained sentiment info


In [5]:
# spaCy + regex setup
nlp = spacy.load("de_core_news_md")
SPLIT = re.compile(r'(?<=[.!?])\s+')

LABELS = {"negative", "neutral", "positive"}

CONTRAST_WORDS = {
    "aber", "jedoch", "doch", "sondern", "allerdings",
    "trotzdem", "dennoch", "hingegen", "immerhin",
    "indessen", "indes", "nichtsdestotrotz", "gleichwohl",
    "obwohl", "obgleich", "obschon", "wenngleich",
    "wiewohl", "auch wenn", "selbst wenn",
    "andererseits", "dagegen", "demgegenüber",
    "im gegensatz", "im gegensatz dazu",
    "im gegenteil", "im unterschied dazu",
    "auf der anderen seite", "auf der einen seite",
    "nur", "zwar", "jedoch nur", "wenigstens",
    "mindestens", "zumindest", "bloß",
    "nicht aber", "wenn auch", "obwohl auch", "nur dass",
    "auch wenn", "wenngleich auch",
    "halt aber", "eigentlich aber",
    "trotz alledem", "trotz allem", "trotz der tatsache",
    "trotz der probleme", "ungeachtet dessen",
    "indes", "indessen", "derweil", "zumal",
    "stellte sich jedoch heraus", "hingegen jedoch",
    "gleichwohl", "nichtsdestoweniger", "ungeachtet",
    "dahingegen", "dies hingegen", "dies jedoch"
}

POS_WORDS = {
    "gut", "toll", "super", "klasse", "genial", "top", "positiv",
    "zufrieden", "zufriedenstellend", "beeindruckend", "hervorragend",
    "ausgezeichnet", "wunderbar", "großartig", "fantastisch",
    "angenehm", "komfortabel", "bequem", "funktioniert gut",
    "preiswert", "günstig", "lohnenswert", "empfehlenswert",
    "solide", "verlässlich", "zuverlässig", "stark", "stabil",
    "effizient", "robust", "gut verarbeitet", "hochwertig",
    "leicht bedienbar", "einfach", "übersichtlich",
    "freundlich", "hilfsbereit", "entgegenkommend",
    "pünktlich", "schnell", "rasch",
    "sehr gut", "extrem gut", "überraschend gut",
    "zufriedenstellend", "akzeptabel"
}

NEG_WORDS = {
    "schlecht", "mangelhaft", "furchtbar", "schrecklich",
    "katastrophal", "grauenhaft", "furchtbar", "mies",
    "negativ", "miserabel", "enttäuschend", "enttäuscht",
    "unzufrieden", "ärgerlich", "frustrierend", "problematisch",
    "kaputt", "defekt", "instabil", "unzuverlässig",
    "langsam", "träge", "teuer", "überteuert",
    "billig verarbeitet", "schlecht verarbeitet",
    "kompliziert", "unverständlich", "chaotisch",
    "fehlerhaft", "buggy", "stürzt ab", "hängt",
    "schwach", "unzureichend", "ungenügend",
    "ärgerlich", "frech", "ineffektiv",
    "schlimm", "unbrauchbar", "wertlos"
}

# Convert label to float score
def label_to_score(lbl: str) -> float:
    lbl = lbl.strip().lower()
    if lbl == "positive": return +1.0
    if lbl == "neutral":  return  0.0
    if lbl == "negative": return -1.0
    return 0.0

# This function checks whether a text contains both positive and negative words
def has_mixed_lexicon(text: str) -> bool:
    doc = nlp(text)

    # Build a single lowercase string of lemmas (spaces and punctuations are not lemmatized)
    lemmas = [tok.lemma_.lower() for tok in doc if not tok.is_punct and not tok.is_space]
    lemma_text = " ".join(lemmas)

    has_pos = any(w in lemma_text for w in POS_WORDS)
    has_neg = any(w in lemma_text for w in NEG_WORDS)
    return has_pos and has_neg

# This function splits a text into segments around contrast words such as "aber" or "jedoch"
def spacy_contrast_segments(text: str) -> List[str]:
     # Use the spaCy pipeline to process the text
    doc = nlp(text)
    segments: List[str] = []

    # Iterate over all sentences
    for sent in doc.sents:
        tokens = list(sent)
        cut_indices = []

        for i, tok in enumerate(tokens): # Iterate over all tokens with their index in the sentence
            if tok.text.lower() in CONTRAST_WORDS and tok.pos_ in {"CCONJ", "SCONJ", "ADV"}: # Check if the token is a known contrast word and is a conjunction word or adverb.
                # left_has_verb = any(t.pos_.startswith("V") for t in tokens[:i]) # Check if the part before the contrast word contains at least one verb
                # right_has_verb = any(t.pos_.startswith("V") for t in tokens[i+1:]) # Check if the part after the contrast word contains at least one verb
                # if left_has_verb and right_has_verb:
                    cut_indices.append(i)

        if not cut_indices:
            seg = sent.text.strip()
            if seg:
                segments.append(seg)
            continue

        # Split by contrast words and add segments that range from one constrast word to the next
        last = 0
        for idx in cut_indices:
            left_tokens = tokens[last:idx]
            if left_tokens:
                seg = sent[left_tokens[0].i : left_tokens[-1].i + 1].text.strip()
                if seg:
                    segments.append(seg)
            last = idx + 1

        # Add last segment that is left after splitting by contrast words
        if last < len(tokens):
            right_tokens = tokens[last:]
            seg = sent[right_tokens[0].i : right_tokens[-1].i + 1].text.strip()
            if seg:
                segments.append(seg)

    return [s for s in segments if s]

# Noun phrase and verb phrase split
def spacy_np_vp_split(text: str) -> List[str]:
    doc = nlp(text)
    sent = next(iter(doc.sents), None) # Split into sentences
    if sent is None:
        return [text.strip()]

    s_doc = sent.as_doc()
    roots = [t for t in s_doc if t.head == t]
    if not roots:
        return [text.strip()]

    # Split
    root = roots[0]
    left_span = s_doc[0:root.i]
    right_span = s_doc[root.i:]

    left = left_span.text.strip()
    right = right_span.text.strip()

    if len(left.split()) >= 2 and len(right.split()) >= 2:
        return [left, right]

    return [text.strip()]

In [23]:
# Smoothing
class EMA:
    def __init__(self, alpha=0.35):
        self.a, self.v = alpha, None

    def reset(self):
        self.v = None

    def update(self, x: float) -> float:
        self.v = x if self.v is None else (self.a * x + (1 - self.a) * self.v)
        return self.v

# Trend reversal detection
class Trend:
    def __init__(self, up_thr=+0.06, down_thr=-0.06, sustain=2):
        self.up_thr = up_thr
        self.down_thr = down_thr
        self.sustain = sustain

        self.state = "flat"       # trend can be "up" or "down" or "flat"
        self.prev = None          # last s_ema (smoothed value)
        self.pending_dir = None   # last not flat desired direction that is accumulating
        self.count = 0            # consecutive steps toward pending_dir

    def reset(self):
        self.state = "flat"
        self.prev = None
        self.pending_dir = None
        self.count = 0

    def update(self, s_ema):
        if self.prev is None:
            self.prev = s_ema
            # first value: nothing to compare against
            return self.state, None, "flat"

        delta = s_ema - self.prev
        self.prev = s_ema

        # decide current desired direction from delta
        if   delta >= self.up_thr:   desired = "up"
        elif delta <= self.down_thr: desired = "down"
        else:                         desired = "flat"

        event = None

        if desired == "flat":
            # no clear movement: reset pending evidence and go flat
            self.pending_dir = None
            self.count = 0
            self.state = "flat"

        elif desired == self.state:
            # already committed to this direction: no new desired direction found
            self.pending_dir = None
            self.count = 0

        else:
            # accumulate only if the direction matches the current pending_dir
            if desired == self.pending_dir:
                self.count += 1
            else:
                self.pending_dir = desired
                self.count = 1

            if self.count >= self.sustain:
                self.state = desired
                self.pending_dir = None
                self.count = 0
                event = f"reversal_to_{desired}"

        return self.state, event, desired

In [24]:
class SentimentTracker:
    def __init__(self, model, tokenizer, device,
                 alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=1):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

        self.ema = EMA(alpha)
        self.trend = Trend(up_thr, down_thr, sustain)
        self.history: list[TurnInfo] = []

    def reset(self):
        self.ema.reset()
        self.trend.reset()
        self.history.clear()

    # Classifier for one segment
    def _classify_label(self, text: str) -> str:
        instruction = (
            "### Instruction:\n"
            "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
            "### Bewertung:\n"
        )
        answer_prefix = "\n\n### Antwort:\n"

        # Build a single lowercase string of lemmas (spaces and punctuations are not lemmatized)
        doc = nlp(text)
        lemmas = [tok.lemma_.lower() for tok in doc if not tok.is_punct and not tok.is_space]
        lemma_text = " ".join(lemmas)

        prompt = instruction + lemma_text + answer_prefix

        self.tokenizer.truncation_side = "left"
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2046,
            padding=False
        ).to(self.device)

        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=2,
                do_sample=False,
                use_cache=False,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        decoded = self.tokenizer.decode(out[0], skip_special_tokens=True)
        answer_part = decoded.split("### Antwort:")[-1] if "### Antwort:" in decoded else decoded
        label = (answer_part.strip().split() or [""])[0].lower()
        return label if label in LABELS else "neutral"

    # segmentation logic
    def _meaningful_segments(self, text: str,
                             max_segments: int = 20) -> List[str]:
        text = text.strip()
        if not text:
            return []

        # First try contrast based segmentation
        candidates = spacy_contrast_segments(text)
        candidates = [c for c in candidates if c.strip()]
        if not candidates:
            return []

        if len(candidates) > 1:
            candidates = candidates[:max_segments]
            labels = [self._classify_label(c) for c in candidates]
            if len(set(labels)) > 1:
                return candidates
            return [text]

        # len(candidates) == 1
        only_seg = candidates[0]

        # Second try noun phrase and verb phrase split
        if has_mixed_lexicon(only_seg):
            np_vp = spacy_np_vp_split(only_seg)
            if len(np_vp) == 2:
                l1 = self._classify_label(np_vp[0])
                l2 = self._classify_label(np_vp[1])
                if l1 != l2:
                    return np_vp

        # Third: no useful split
        return [text]

    # full sentiment analysis for one turn
    def _analyze_sentiment(self, text: str,
                           max_sents: int = 12) -> Dict:
        text = text.strip()
        if not text:
            return {
                "segments": [],
                "overall_score": 0.0,
                "overall_label": "neutral",
            }

        raw_sents = [s.strip() for s in SPLIT.split(text) if s.strip()]
        if not raw_sents:
            raw_sents = [text]

        all_segments: List[str] = []
        for sent in raw_sents[:max_sents]:
            segs = self._meaningful_segments(sent)
            if not segs:
                continue
            all_segments.extend(segs)

        if not all_segments:
            all_segments = [text]

        segment_infos: List[SegmentInfo] = []
        scores: List[float] = []
        lengths: List[int] = []

        for seg in all_segments:
            lbl = self._classify_label(seg)
            sc = label_to_score(lbl)
            L = len(nlp(seg)) # Number of tokens (words and symbols like ?)

            segment_infos.append(SegmentInfo(
                text=seg,
                label=lbl,
                score=sc,
                length=L,
            ))

            scores.append(sc)
            lengths.append(L)

        total_len = sum(lengths) or 1
        overall_score = sum(sc * L for sc, L in zip(scores, lengths)) / total_len

        if overall_score > 0.2:
            overall_label = "positive"
        elif overall_score < -0.2:
            overall_label = "negative"
        else:
            if any(seg.label == "negative" for seg in segment_infos):
                overall_label = "negative"
            elif any(seg.label == "positive" for seg in segment_infos):
                overall_label = "positive"
            else:
                overall_label = "neutral"

        return {
            "segments": segment_infos,
            "overall_score": overall_score,
            "overall_label": overall_label,
        }

    # one conversation turn
    def step(self, who: str, text: str) -> TurnInfo:
        # get detailed sentiment for this turn
        result = self._analyze_sentiment(text)
        raw = result["overall_score"]
        overall_label = result["overall_label"]
        segments = result["segments"]  # list[SegmentInfo]

        # update EMA & trend tracker
        sm = self.ema.update(raw)
        tr, ev, desired = self.trend.update(sm)

        # pack everything into TurnInfo
        info = TurnInfo(
            who=who,
            text=text,
            s_raw=raw,
            s_ema=sm,
            trend=tr,
            event=ev,
            desired=desired,
            overall_label=overall_label,
            segments=segments,
        )
        self.history.append(info)
        return info

In [8]:
device = "cuda" # Used in colab

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-sentiment-model-guhr-benchmark"
subfolder = "adapters/epoch_003"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [9]:
# 4-bit quantization config
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

# Load base model in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quant_config,
    device_map="auto",
)

# Attach LoRA adapters
lora_model = PeftModel.from_pretrained(base_model, lora_repo_id, subfolder=subfolder)
#lora_model = PeftModel.from_pretrained(base_model, lora_repo_id)

lora_model = lora_model.to(device)
lora_model.eval()

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapters/epoch_003/adapter_model.safeten(…):   0%|          | 0.00/61.3M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.058, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Lin

In [10]:
# Load Qwen model (4-bit)
ID = "Qwen/Qwen2.5-1.5B-Instruct"
# ID = "LeoLM/leo-hessianai-7b-chat"
use_4bit = torch.cuda.is_available()
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
) if use_4bit else None

q_tok = AutoTokenizer.from_pretrained(ID, use_fast=True)
q_mdl = AutoModelForCausalLM.from_pretrained(
    ID,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    quantization_config=bnb
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [11]:
# Generate the reply of the chatbot
def bot_reply(history, user_text, max_new_tokens=160):
    msgs = history + [{"role": "user", "content": user_text}]
    prompt = q_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    x = q_tok(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        x = {k: v.to(q_mdl.device) for k, v in x.items()}
    y = q_mdl.generate(**x, max_new_tokens=max_new_tokens, do_sample=False,
                       eos_token_id=q_tok.eos_token_id, pad_token_id=q_tok.eos_token_id)
    gen = q_tok.decode(y[0][x["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return gen

In [44]:
def build_style_instruction(info) -> str:
    event = getattr(info, "event", None)
    trend = getattr(info, "trend", None)
    label = getattr(info, "overall_label", None)

    # Priorität: event > trend > label
    if event == "reversal_to_down":
        mode = "reversal_to_down"
    elif event == "reversal_to_up":
        mode = "reversal_to_up"
    elif trend == "down":
        mode = "down"
    elif trend == "up":
        mode = "up"
    elif label == "negative":
        mode = "negative"
    elif label == "positive":
        mode = "positive"
    else:
        mode = "neutral"

    common = (
        "STIL-ANWEISUNG (nur Ton; Inhalt strikt nach Turn-Plan aus BASE_SYS):\n"
        "- Du MUSST das Format einhalten: 1 Einleitungssatz (max. 12 Wörter), dann GENAU 3 Bulletpoints.\n"
        "- Keine zusätzlichen Absätze nach den Bulletpoints.\n"
        "- Kurze Sätze. Keine Rechtfertigungen. Keine Schuldzuweisungen.\n"
        "- Wiederhole keine emotionalen Phrasen mehrfach.\n"
    )

    prompts = {}

    prompts["neutral"] = common + (
        "MODE: neutral\n"
        "- Einleitungssatz neutral, ohne Emotion (z.B. „Alles klar, ich helfe Ihnen dabei.“).\n"
        "- Keine Entschuldigung, keine Validierung.\n"
    )

    prompts["positive"] = common + (
        "MODE: positive\n"
        "- Einleitungssatz MUSS 1 positives/kooperatives Signal enthalten: "
        "„gut“, „danke“, oder „prima“ (z.B. „Gut, danke – so kommen wir weiter.“).\n"
        "- Freundlich, aber nicht überschwänglich.\n"
    )

    prompts["up"] = common + (
        "MODE: up\n"
        "- Einleitungssatz mild bestärkend (z.B. „Okay, das wirkt schon besser.“).\n"
        "- Ruhig stabilisieren, nicht zu emotional.\n"
    )

    prompts["reversal_to_up"] = common + (
        "MODE: reversal_to_up\n"
        "- Einleitungssatz MUSS die positive Wendung spiegeln (z.B. „Sehr gut, das ist ein Fortschritt.“).\n"
        "- Abschlussorientiert.\n"
    )

    prompts["negative"] = common + (
        "MODE: negative\n"
        "- Einleitungssatz MUSS genau 1 dieser Wörter enthalten: „ärgerlich“ ODER „frustrierend“.\n"
        "- Danach sofort zur Lösung, ohne weitere Emotionalisierung.\n"
        "- Keine harten Wörter wie „inakzeptabel“, „selbst schuld“.\n"
    )

    prompts["down"] = common + (
        "MODE: down\n"
        "- Einleitungssatz MUSS „ich verstehe“ enthalten (z.B. „Ich verstehe – das ist frustrierend.“).\n"
        "- Deeskalierend, ruhig, kontrolliert.\n"
        "- Bulletpoints als klare Schritte, kein Diskutieren.\n"
    )

    prompts["reversal_to_down"] = common + (
        "MODE: reversal_to_down\n"
        "- Einleitungssatz MUSS enthalten: „Das ist ärgerlich“ UND „wir klären das“.\n"
        "  Beispiel: „Das ist ärgerlich – wir klären das jetzt gemeinsam.“\n"
        "- Bulletpoints in „Jetzt / Als Nächstes / Danach“ Struktur.\n"
        "- Sehr beruhigend, sehr konkret.\n"
    )

    return prompts[mode]

In [45]:
BASE_SYS = [{
  "role": "system",
  "content": """
Du bist Kundenservice eines Online-Versandhändlers und hilfst bei Lieferproblemen.

Fallkontext (fest):
- Kunde in Klagenfurt (AT), Standardversand.
- Liefertermin: gestern (08:00–18:00).
- Tracking: „Zugestellt“.
- Keine Unterschrift, kein Foto, kein dokumentierter Ablageort.

ALLGEMEINE REGELN:
- Antworte immer auf Deutsch.
- Antworte kurz, klar, lösungsorientiert.
- Keine echten persönlichen Daten anfordern (kein Name, keine Bestellnummer, keine genaue Adresse).
- Maximal 3 Bulletpoints.
- Keine erfundenen Details (keine Uhrzeiten/Hotlines/Entschädigungen).

WIEDERHOLUNGS-REGELN (sehr wichtig):
- Verwende keinen Bulletpoint, der bereits in einer früheren Antwort wortgleich vorkam.
- Wiederhole Maßnahmen nicht als Hauptpunkte. Wenn nötig, verweise kurz („wie erwähnt“) und nenne dann etwas Neues.
- Keine Dopplungen innerhalb derselben Antwort.
- Jede Antwort muss mindestens 1 neuen Aspekt enthalten.

ANTWORTFORMAT (immer):
- 1 kurzer Einleitungssatz (max. 12 Wörter).
- Dann GENAU 3 Bulletpoints.
- Keine zusätzlichen Absätze nach den Bulletpoints.
- Maximal 1 Rückfrage und nur in Turn 4.

TURN-PLAN (Inhalte, nicht Ton) – nutze pro Turn neue Schwerpunkte:

Turn 1 (Status):
- Einordnung „Zugestellt“ (Status ≠ sichere Übergabe)
- 1 plausible Ursache: Ablageort ODER Nachbarannahme
- 1 Sofort-Check: Hausflur/Türbereich/Paketbox

Turn 2 („zugestellt“, aber fehlt):
- 2. Ursache (anders als Turn 1): Scan-Fehler ODER falsche Tür im Gebäude
- Hinweis: Standardversand ohne Unterschrift
- Check: direkte Nachbarn + Hauseingang/Empfangsbereich (falls vorhanden)

Turn 3 („Was zuerst prüfen?“):
- Priorisierte Reihenfolge aus 3 Checks (nicht wortgleich zu Turn 1/2)
- Ergänze 1 neuen Ort: z.B. Abstellraum, Hausmeister/Verwaltung (nur allgemein)
- Hinweis: kurze Verzögerung trotz Status möglich (ohne Zeitversprechen)

Turn 4 („Schritt-für-Schritt“):
- Exakte 3-Schritt-Anleitung (1/2/3) in klarer Reihenfolge
- Danach: Nachforschung anstoßen, wenn erfolglos
- Erlaubt: 1 Rückfrage, ob Nachbarn schon gefragt wurden

Turn 5 („dringend, schnell“):
- Schnellste Aktion: Nachbarannahme sofort klären (kurz)
- Parallel: Nachforschung starten, wenn nicht auffindbar
- Erwartungsmanagement: Rückmeldung typischerweise in wenigen Werktagen (ohne Garantie)

Turn 6 („wenn heute nicht auftaucht“):
- Prozess: Nachforschung → Prüfung → Ergebnis
- Ergebnisoptionen: Ersatzlieferung ODER Rückerstattung bei bestätigtem Verlust
- Klarer nächster Schritt (ohne persönliche Daten)
"""
}]

def make_trackers():
    t = SentimentTracker(lora_model, tokenizer, device, alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=2)
    tb = SentimentTracker(lora_model, tokenizer, device, alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=2)
    try:
        t.reset(); tb.reset()
    except Exception:
        pass
    return t, tb

In [49]:
def chat_a(user_text, chat_ui, qwen_hist):
    user_text = (user_text or "").strip()
    if not user_text:
        return chat_ui, qwen_hist, ""

    chat_ui = chat_ui or []        # messages for UI
    qwen_hist = qwen_hist or []    # messages for LLM history (system+user+assistant)

    reply = bot_reply(qwen_hist, user_text)

    # UI history in messages format
    chat_ui = chat_ui + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": reply},
    ]

    # LLM history in messages format
    qwen_hist = qwen_hist + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": reply},
    ]

    return chat_ui, qwen_hist, ""

from datetime import datetime

def chat_b(user_text, chat_ui, qwen_hist, tracker, tracker_bot, syslog):
    user_text = (user_text or "").strip()
    if not user_text:
        return chat_ui, qwen_hist, tracker, tracker_bot, syslog, ""

    chat_ui = chat_ui or []
    qwen_hist = qwen_hist or []
    syslog = syslog or []

    style_instruction = ""
    info = None
    if tracker is not None:
        info = tracker.step("human", user_text)
        style_instruction = (build_style_instruction(info) or "").strip()

    # Log style system message (even if empty)
    user_turn_index = sum(1 for m in qwen_hist if isinstance(m, dict) and m.get("role") == "user") + 1
    syslog = syslog + [{
        "turn_index": user_turn_index,
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "style_instruction": style_instruction,
        "tracker_event": getattr(info, "event", None) if info is not None else None,
        "tracker_trend": getattr(info, "trend", None) if info is not None else None,
        "tracker_label": getattr(info, "overall_label", None) if info is not None else None,
    }]

    # Prepend style instruction ONLY for generation
    local_hist = ([{"role": "system", "content": style_instruction}] if style_instruction else []) + qwen_hist
    reply = bot_reply(local_hist, user_text)

    if tracker_bot is not None:
        tracker_bot.step("bot", reply)

    # UI history (messages) - we DO NOT show the style system prompt in the UI by default
    chat_ui = chat_ui + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": reply},
    ]

    # Persist only user/assistant in qwen history (keeps base prompt clean)
    qwen_hist = qwen_hist + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": reply},
    ]

    return chat_ui, qwen_hist, tracker, tracker_bot, syslog, ""

def reset_b():
    t, tb = make_trackers()
    return [], BASE_SYS.copy(), t, tb, [], ""

In [50]:
import json, time
from datetime import datetime
import os

OUT_PATH = "/mnt/data/study_submissions.jsonl"

def segment_to_dict(seg):
    return {
        "text": seg.text,
        "label": seg.label,
        "score": float(seg.score),
        "length": int(seg.length),
    }

def turn_to_dict(t):
    return {
        "who": t.who,
        "text": t.text,
        "s_raw": float(t.s_raw),
        "s_ema": float(t.s_ema),
        "trend": t.trend,
        "event": t.event,
        "desired": t.desired,
        "overall_label": t.overall_label,
        "segments": [segment_to_dict(s) for s in (t.segments or [])],
    }

def tracker_to_dict(tr):
    if tr is None:
        return None
    return {
        "ema": {
            "alpha": float(getattr(tr.ema, "a", 0.0)),
            "v": None if getattr(tr.ema, "v", None) is None else float(tr.ema.v),
        },
        "trend": {
            "up_thr": float(getattr(tr.trend, "up_thr", 0.0)),
            "down_thr": float(getattr(tr.trend, "down_thr", 0.0)),
            "sustain": int(getattr(tr.trend, "sustain", 0)),
            "state": getattr(tr.trend, "state", None),
            "prev": None if getattr(tr.trend, "prev", None) is None else float(tr.trend.prev),
            "pending_dir": getattr(tr.trend, "pending_dir", None),
            "count": int(getattr(tr.trend, "count", 0)),
        },
        "history": [turn_to_dict(t) for t in (tr.history or [])],
    }

def submit_questionnaire(q1, q2, q3,
                         chat_a_ui, chat_b_ui,
                         a_qwen_hist, b_qwen_hist,
                         tracker_human, tracker_bot,
                         b_style_syslog):
    try:
        record = {
            "timestamp_utc": datetime.utcnow().isoformat() + "Z",
            "questionnaire": {"q1": q1, "q2": q2, "q3": q3},

            # Visible chats (NOW messages)
            "chatA_ui": chat_a_ui or [],
            "chatB_ui": chat_b_ui or [],

            # Prompt histories (role/content dicts)
            "chatA_qwen_history": a_qwen_hist or [],
            "chatB_qwen_history": b_qwen_hist or [],

            # NEW: injected style system prompts (per turn)
            "chatB_style_system_log": b_style_syslog or [],

            "sentiment_tracker_human": tracker_to_dict(tracker_human),
            "sentiment_tracker_bot": tracker_to_dict(tracker_bot),
        }

        os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
        with open(OUT_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

        return "Saved.", json.dumps(record, ensure_ascii=False, indent=2)

    except Exception as e:
        return f"Submit failed: {type(e).__name__}: {e}", ""

In [51]:
with gr.Blocks() as demo:
    # states
    a_ui_state = gr.State([])                 # messages list for UI
    a_qwen_state = gr.State(BASE_SYS.copy())  # messages list for LLM history

    b_ui_state = gr.State([])
    b_qwen_state = gr.State(BASE_SYS.copy())
    tracker_state = gr.State(None)
    tracker_bot_state = gr.State(None)

    # NEW: syslog state for injected system prompts
    b_syslog_state = gr.State([])

    with gr.Tabs():
        with gr.Tab("Chatbot A"):
            chatA = gr.Chatbot(height=420)  # Gradio 6 expects messages format by default
            msgA = gr.Textbox(label="Message")
            errA = gr.Markdown("")
            sendA = gr.Button("Send")

            sendA.click(
                chat_a,
                [msgA, a_ui_state, a_qwen_state],
                [chatA, a_qwen_state, errA]
            ).then(lambda: "", None, msgA)

        with gr.Tab("Chatbot B (sentiment-aware)"):
            chatB = gr.Chatbot(height=420)
            msgB = gr.Textbox(label="Message")
            errB = gr.Markdown("")
            with gr.Row():
                sendB = gr.Button("Send")
                resetB_btn = gr.Button("Reset")

            def init_trackers(tr, trb):
                if tr is None or trb is None:
                    return make_trackers()
                return tr, trb

            demo.load(init_trackers, [tracker_state, tracker_bot_state], [tracker_state, tracker_bot_state])

            sendB.click(
                chat_b,
                [msgB, b_ui_state, b_qwen_state, tracker_state, tracker_bot_state, b_syslog_state],
                [chatB, b_qwen_state, tracker_state, tracker_bot_state, b_syslog_state, errB]
            ).then(lambda: "", None, msgB)

            resetB_btn.click(
                reset_b,
                None,
                [chatB, b_qwen_state, tracker_state, tracker_bot_state, b_syslog_state, errB]
            ).then(lambda: [], None, b_ui_state)

        with gr.Tab("Questionnaire"):
            q1 = gr.Textbox(label="Q1")
            q2 = gr.Textbox(label="Q2")
            q3 = gr.Textbox(label="Q3")
            submit = gr.Button("Submit")
            status = gr.Markdown("")
            returned_json = gr.Code(label="Returned data", language="json")

            submit.click(
                submit_questionnaire,
                [
                    q1, q2, q3,
                    a_ui_state, b_ui_state,
                    a_qwen_state, b_qwen_state,
                    tracker_state, tracker_bot_state,
                    b_syslog_state
                ],
                [status, returned_json]
            )

demo.queue(default_concurrency_limit=1, max_size=30)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://27d6a3460f062f094d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipython-input-2711398593.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 763, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2117, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/li

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://27d6a3460f062f094d.gradio.live
